### 1. Карточки вакансий

H0: mu (среднее генеральной совокупности) = 120 сек

In [1]:
import numpy as np

n = 100
mu = 120
mean_x = 124
sigma = 25
se = sigma/np.sqrt(n)

print(f"Выборочное среднее ушло от среднего на {(mean_x - mu) / se:.3f} стандартных ошибок.")

Выборочное среднее ушло от среднего на 1.600 стандартных ошибок.


Оснований отвергать H0 нет. Для уровня значимости alpha = 0.05, отвергаем гипотезу, когда разница больше 1.96 стандартных ошибок (z-test). В нашем случае отклонение может быть шумом.

### 2. Доверительный интервал

In [2]:
from scipy.stats import t

n1 = 40
x1 = 50
s1 = 10

n2 = 40
x2 = 55
s2 = 10

alpha = 0.05

se_diff = np.sqrt((s1 ** 2) / n1 + (s2 ** 2) / n2)
print(f"Разница: {x2 - x1}")
df = n1 + n2 - 2
print("Степень свободы:", df)
t_score = t.ppf(1 - alpha/2, df, loc=0, scale=1)
print(f"t-score: {t_score:.2f}")
print(f"95% ДИ: ({x2 - x1 - t_score * se_diff:.3f}, {x2 - x1 + t_score * se_diff:.3f})")



Разница: 5
Степень свободы: 78
t-score: 1.99
95% ДИ: (0.548, 9.452)


0 не попадает в ДИ, значит можно с уверенностью в 95% отвергнуть гипотезу, что средние выборок равны. Самый маленький ожидаемый эффект, с которым гипотеза не будет опровергаться - это 0.55.

### 3. Z-test / тест Уилсона для конверсий

H0: mu1 = mu2 (конверсиии обеих групп равны).

In [3]:
from scipy.stats import norm

sample_size = 1000

k1 = 90
k2 = 120

x1 = k1 / sample_size # Конверсия группы 1
x2 = k2 / sample_size # Конверсия группы 2
print(f"Конверсия выборки 1: {x1:.2f}")
print(f"Конверсия выборки 2: {x2:.2f}")
p_pooled= (k1 + k2) / (2 * sample_size)
print(f"Общая конверсия: {p_pooled:.2f}")
print(f"Отклонение: {(x2 - x1) * 100:.2f} п.п.")
se_diff = np.sqrt(p_pooled*(1-p_pooled)*(2/sample_size))
print(f"Стандартная ошибка разности: {se_diff * 100:.2f} п.п.")

z_statistics = (x2 - x1) / se_diff
print(f"z-статистика: {z_statistics:.4f}")

print(f"p-value: {(norm.sf(abs(z_statistics)))*2:.3f}")

Конверсия выборки 1: 0.09
Конверсия выборки 2: 0.12
Общая конверсия: 0.10
Отклонение: 3.00 п.п.
Стандартная ошибка разности: 1.37 п.п.
z-статистика: 2.1883
p-value: 0.029


In [4]:
from statsmodels.stats.proportion import test_proportions_2indep

sample_size = 1000

k1 = 90
k2 = 120

result = test_proportions_2indep(count1=k1, nobs1=sample_size,
                        count2=k2, nobs2=sample_size,
                        compare="diff", method="score")

print(f"z-статистика: {result.statistic:.4f}")
print(f"p-value: {result.pvalue:.3f}")

z-статистика: -2.1877
p-value: 0.029


P-value меньше стандартного уровня значимости alpha = 0.05. Значит опровергаем гипотезу.

### 4. Распределение p-value без эффекта

In [5]:
from scipy.stats import ttest_ind
import plotly.express as px

experiments_cnt = 2000
sample_size = 200
rng = np.random.default_rng(42)

pvalues = []
for exp_ind in range(experiments_cnt):
    groupA = rng.normal(0, 1, size=sample_size)
    groupB = rng.normal(0, 1, size=sample_size)
    result = ttest_ind(groupA, groupB, equal_var=False, alternative="two-sided")
    pvalues.append(result.pvalue)
    
fig = px.histogram(x=pvalues, title="Pvalues distribution")

fig.show()

P-value распределены равномерно. Эффекта нет, поэтому каждое значение одинаково вероятно.

### 5. Тест Уэлча и Манна-Уитни для групп с эффектом

In [6]:
from scipy import stats

experiments_cnt = 1000
sample_size = 60
rng = np.random.default_rng(42)

t = []
t_pvalues = []

u = []
u_pvalues = []

for exp_ind in range(experiments_cnt):
    groupA = rng.normal(0, 1, size=sample_size)
    groupB = rng.normal(1, 1, size=sample_size)
    groupB[-1] = -50
    groupB[-2] = -50
    ttest_result = ttest_ind(groupA, groupB, equal_var=False, alternative="two-sided")
    mw_result = stats.mannwhitneyu(groupA, groupB, alternative="two-sided")

    t.append(ttest_result.statistic)
    t_pvalues.append(ttest_result.pvalue)

    u.append(mw_result.statistic)
    u_pvalues.append(mw_result.pvalue)
    
fig = px.histogram(x=t, title="t-statistic distribution")
fig.show()

fig = px.histogram(x=u, title="U-statistic distribution")
fig.show()

t_pvalues = np.array(t_pvalues)
u_pvalues = np.array(u_pvalues)

fig = px.histogram(x=t_pvalues, title="T-test pvalues distribution")
fig.add_vline(x=0.05)
fig.show()

fig = px.histogram(x=u_pvalues, title="Mann-Whitney pvalues distribution")
fig.add_vline(x=0.05)
fig.show()

print(f"При T-test отвергается {t_pvalues[t_pvalues < 0.05].size/experiments_cnt:.2%} случаев")
print(f"При тесте Манна-Уитни отвергается {u_pvalues[u_pvalues < 0.05].size/experiments_cnt:.2%} случаев")

При T-test отвергается 0.00% случаев
При тесте Манна-Уитни отвергается 99.90% случаев


Критерий Манна-Уитни справляется с нахождением различий в выборках, тогда как t-test нет. Два низких значения увеличивают дисперсию, из-за чего увеличивается и стандартная ошибка, которую использует t-test. А Манн-Уитни использует ранги, нам не важно насколько далеки крайние значения. Поэтому в данном случае Манн-Уитни отвергает 0 гипотезу в 99.9% экспериментов, а тест Уэлча почти ни разу.

### 6. Доверительный интервал Уэлча

In [7]:
from scipy.stats import t

n1 = 80
n2 = 75
rng = np.random.default_rng(42)

groupA = rng.normal(10, 2, size=n1)
groupB = rng.normal(10.8, 2.6, size=n2)

se_diff = np.sqrt((groupA.std(ddof=1) ** 2)/n1 + (groupB.std(ddof=1) ** 2)/n2)
df = n1 + n2 - 2 
alpha = 0.05
t_score = t.ppf(1-alpha/2, df)

mean_diff = groupB.mean() - groupA.mean()
t_statistics = mean_diff / se_diff
pvalue = (1 - t.cdf(abs(t_statistics), df))*2

print(f"Отклонение: {mean_diff:.4f}")
print(f"95% ДИ: ({mean_diff - t_score * se_diff:.3f}, {mean_diff + t_score * se_diff:.3f})")
print(f"t-статистика: {t_statistics:.3f}, pvalue: {pvalue:.3f}")

Отклонение: 0.2960
95% ДИ: (-0.353, 0.945)
t-статистика: 0.901, pvalue: 0.369


In [8]:
from scipy.stats import ttest_ind

result = ttest_ind(groupB, groupA,
                            equal_var=False, 
                            alternative="two-sided")

mean_diff = groupB.mean() - groupA.mean()
ci = result.confidence_interval(confidence_level=0.95)

print(f"Отклонение: {mean_diff:.4f}")
print(f"95% ДИ: ({ci.low:.3f}, {ci.high:.3f})")
print(f"t-статистика: {result.statistic:.3f}, pvalue: {result.pvalue:.3f}")

Отклонение: 0.2960
95% ДИ: (-0.354, 0.946)
t-статистика: 0.901, pvalue: 0.369


0 попадает в ДИ, pvalue больше уровня значимости 0.05, значит H0 не отвергаем. Самый минимальный возможный эффект при верной H0 равен 0.

### 7. Бутстрап

In [10]:
experiments_cnt = 2000

rng = np.random.default_rng(42)

means = []
for exp_ind in range(experiments_cnt):
    bootstrapA = rng.choice(groupA, groupA.size, replace=True)
    bootstrapB = rng.choice(groupB, groupB.size, replace=True)

    means.append(bootstrapB.mean() - bootstrapA.mean())

means = np.array(means)
fig = px.histogram(x=means, title="Распределение разниц средних")

fig.show()

p_2_5, p_97_5 = np.percentile(means, [2.5, 97.5])
print(f"95% ДИ: ({p_2_5:.3f}, {p_97_5:.3f})")
print(f"Доля muA >= muB: {means[means <= 0].size / experiments_cnt:.3f}")

95% ДИ: (-0.326, 0.939)
Доля muA >= muB: 0.173


Границы бутстрап ДИ похожи на границы ДИ Уэлча. 0 входит в ДИ, значит H0 не опровергаем. Доля экспериментов, когда разность средних не больше 0 не является p-value. Наша H0 заключается в том, что средние между совокупностями равны. 

Получившееся распределение разниц средних центрировано вокруг разницы исходных выборок, а не 0 (см. график). Оно показывает, насколько сильно наблюдаемая разница может колебаться из-за случайности выборок. А доля экспериментов оценивает, насколько правдоподобно, что истинная разница средних не положительна.